In [ ]:
import os
from typing import TypedDict
from dotenv import load_dotenv , find_dotenv
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph,START,END

load_dotenv(find_dotenv() , override=True)

model = ChatOpenAI(
    model = os.getenv("KIMI_MODEL"),
    base_url = os.getenv("KIMI_BASE_URL").rstrip("/") + "/v1",
    api_key = "not-needed",
    default_headers={
        "Modal-Key": os.getenv("MODEL_KEY"),
        "Modal-Secret": os.getenv("MODEL_SECRET"),
    },
)


#state
class Guide(TypedDict):
    q: float
    a: str



#node
def protien(state: Guide) -> Guide:
    bmi = state['q']
    quest = f"suggest protien intake according to the bmi, bmi is {bmi}"
    ans = model.invoke(quest).content
   
    return {"a": ans}

#graph
graph = StateGraph(Guide)
#add node
graph.add_node("func", protien)
#add edges
graph.add_edge(START,"func")
graph.add_edge("func",END)
#compile
workflow = graph.compile()

#invoke

initail_state = {"q": 22.4}

if __name__ == "__main__":
    final_state = workflow.invoke(initail_state)
    print(final_state)